# 06 — Atención CBAM (Convolutional Block Attention Module)

**Qué hace:** define las capas `channel_attention` y `spatial_attention`, construye un modelo MobileNetV3Small + CBAM + cabeza densa, lo entrena (fase 1, backbone congelado), lo guarda, lo vuelve a cargar con `custom_objects` (necesario por las capas custom) y hace una segunda ronda de fine-tuning descongelando el 20% final del backbone, con evaluación final.

**Consume:** `asl_split_nuevo/train|test|validate/<clase>/*` (producido por `02_split_por_grupos.ipynb`).

**Produce:** `model_cbam_my_tf_fine_tuned_.keras` (modelo tras la fase 1) y `model_cargado_cbam_my_tf_fine_tuned_final_.keras` (modelo tras el fine-tuning).

## Imports y rutas base

In [ ]:
from pathlib import Path
from tensorflow.keras.utils import image_dataset_from_directory


## Definición de `channel_attention` y `spatial_attention`

In [41]:
### definir dos capas de atención (channel y spatial attention)
import tensorflow as tf
from tensorflow.keras import layers

@tf.keras.utils.register_keras_serializable()
class channel_attention(layers.Layer):
    def __init__(self, ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio})
        return config

    def build(self, input_shape):
        channels =  input_shape[-1]
        self.reduce = layers.Dense(channels // self.ratio, activation='relu')
        self.expand = layers.Dense(channels)

    def call(self, x):
        avg = tf.reduce_mean(x, axis=[1,2])
        max = tf.reduce_max(x, axis=[1,2])
        weights = tf.sigmoid(
            self.expand(self.reduce(avg)) +
            self.expand(self.reduce(max))
        )
        weights = tf.reshape(weights, [-1, 1, 1, tf.shape(x)[-1]])
        return x * weights
        
    



In [40]:

@tf.keras.utils.register_keras_serializable()
class spatial_attention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size        # retener el parámetro
        self.conv = layers.Conv2D(1, kernel_size, padding='same', activation='sigmoid')

    def get_config(self):
        config = super().get_config()
        config.update({'kernel_size': self.kernel_size})   # la receta, no el objeto
        return config


    def call(self, x):
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        max = tf.reduce_max(x, axis=-1, keepdims=True)
        map = self.conv(tf.concat([avg, max], axis=-1))
        return x * map
    
    

## Carga de datos y data augmentation

In [50]:
DESTINO = Path('./asl_split_nuevo')   
train_dir = DESTINO / 'train'
test_dir = DESTINO / 'test'
val_dir = DESTINO / 'validate'


trained_generated = image_dataset_from_directory(
    train_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

test_generated = image_dataset_from_directory(
    test_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

val_generated = image_dataset_from_directory(
    val_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

data_augmentation = tf.keras.Sequential([
   ## tf.keras.layers.RandomRotation(0.15),
    ##tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

Found 8620 files belonging to 36 classes.
Found 14517 files belonging to 36 classes.
Found 12267 files belonging to 36 classes.


## Backbone MobileNetV3Small congelado + bloques CBAM

In [11]:
pre_trained = tf.keras.applications.MobileNetV3Small(
    weights='imagenet',
    include_top = False,
    input_shape=(224, 224, 3)
)

pre_trained.trainable = False

In [15]:
inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
##x = pre_trained(inputs, training = False)
x = pre_trained(x, training = False)
x = channel_attention(ratio=16)(x)           # qué canales
x = spatial_attention(kernel_size=7)(x)      # dónde
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(224, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(len(trained_generated.class_names), activation='softmax')(x)

## kernel_regularizer=tf.keras.regularizers.l2(1e-5)

model_cbam = tf.keras.Model(inputs, x)

model_cbam.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=0.001), loss="categorical_crossentropy", metrics=['accuracy', 'precision', 'recall'])

In [16]:
earlyStopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True 
)


## Entrenamiento (fase 1)

In [17]:
model_cbam.fit(
    trained_generated,
    epochs=30,
    validation_data = val_generated,
    callbacks=[earlyStopping]
)

Epoch 1/30


/Users/denilsonbarahona/Documents/IA/model-sign-language/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


87/87 ━━━━━━━━━━━━━━━━━━━━ 163s 2s/step - accuracy: 0.6107 - loss: 1.6024 - precision: 0.9586 - recall: 0.3597 - val_accuracy: 0.5419 - val_loss: 1.4566 - val_precision: 0.8089 - val_recall: 0.3584
Epoch 2/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.9354 - loss: 0.3038 - precision: 0.9788 - recall: 0.8770 - val_accuracy: 0.7136 - val_loss: 0.9137 - val_precision: 0.8274 - val_recall: 0.6045
Epoch 3/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.9800 - loss: 0.1200 - precision: 0.9908 - recall: 0.9628 - val_accuracy: 0.7666 - val_loss: 0.7211 - val_precision: 0.8434 - val_recall: 0.7094
Epoch 4/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 216s 2s/step - accuracy: 0.9884 - loss: 0.0718 - precision: 0.9936 - recall: 0.9802 - val_accuracy: 0.8044 - val_loss: 0.5883 - val_precision: 0.8563 - val_recall: 0.7605
Epoch 5/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 160s 2s/step - accuracy: 0.9937 - loss: 0.0414 - precision: 0.9959 - recall: 0.9897 - val_accuracy: 0.8282 - val_loss: 0.5502 - val_preci

## Evaluación y guardado

In [19]:
evaluatex = model_cbam.evaluate(test_generated)
print(evaluatex)

146/146 ━━━━━━━━━━━━━━━━━━━━ 90s 620ms/step - accuracy: 0.8436 - loss: 0.6554 - precision: 0.8661 - recall: 0.8292
[0.6553962230682373, 0.8436316251754761, 0.8660958409309387, 0.8291658163070679]


In [20]:
model_cbam.save('model_cbam_my_tf_fine_tuned_.keras')

## Carga del modelo guardado (con `custom_objects`) y descongelado del 20% final

In [29]:
model_cargado = tf.keras.models.load_model('model_cbam_my_tf_fine_tuned_.keras',  custom_objects={
        'channel_attention': channel_attention,
        'spatial_attention': spatial_attention,
    })

print(model_cargado.layers[2])
base = model_cargado.layers[2]   # ajusta el índice al que viste en el print
print(f"La base tiene {len(base.layers)} capas internas")


base.trainable = True

total_layers = len(base.layers)
limit = int(total_layers * 0.80)

for layer in base.layers[:limit]:
    layer.trainable = False

print(f"Congeladas: {limit} | Entrenables: {total_layers - limit}")

model_cargado.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)


<Functional name=MobileNetV3Small, built=True>
La base tiene 157 capas internas
Congeladas: 125 | Entrenables: 32


## Fine-tuning: carga de datos y entrenamiento

In [30]:
from pathlib import Path
DESTINO = Path('./asl_split_nuevo')   
train_dir = DESTINO / 'train'
test_dir = DESTINO / 'test'
val_dir = DESTINO / 'validate'



trained_generated = image_dataset_from_directory(
    train_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

test_generated = image_dataset_from_directory(
    test_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

val_generated = image_dataset_from_directory(
    val_dir,
    labels='inferred',
    batch_size = 100,
    color_mode='rgb',
    image_size=(224, 224),
    label_mode="categorical",
    pad_to_aspect_ratio=True,
)

earlyStopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True 
)

Found 8620 files belonging to 36 classes.
Found 14517 files belonging to 36 classes.
Found 12267 files belonging to 36 classes.


In [31]:
history_cbam_ft = model_cargado.fit(
    trained_generated,
    epochs=30,
    validation_data=val_generated,
    callbacks=[earlyStopping]
)

Epoch 1/30


/Users/denilsonbarahona/Documents/IA/model-sign-language/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


87/87 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.8991 - loss: 0.4722 - precision: 0.9224 - recall: 0.8820 - val_accuracy: 0.6235 - val_loss: 1.5794 - val_precision: 0.6708 - val_recall: 0.6029
Epoch 2/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.9910 - loss: 0.0467 - precision: 0.9939 - recall: 0.9871 - val_accuracy: 0.6620 - val_loss: 1.4356 - val_precision: 0.7044 - val_recall: 0.6440
Epoch 3/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 131s 2s/step - accuracy: 0.9962 - loss: 0.0230 - precision: 0.9969 - recall: 0.9954 - val_accuracy: 0.6869 - val_loss: 1.3463 - val_precision: 0.7223 - val_recall: 0.6689
Epoch 4/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 130s 2s/step - accuracy: 0.9977 - loss: 0.0148 - precision: 0.9983 - recall: 0.9970 - val_accuracy: 0.7185 - val_loss: 1.2424 - val_precision: 0.7475 - val_recall: 0.7028
Epoch 5/30
87/87 ━━━━━━━━━━━━━━━━━━━━ 134s 2s/step - accuracy: 0.9987 - loss: 0.0106 - precision: 0.9991 - recall: 0.9980 - val_accuracy: 0.7447 - val_loss: 1.1242 - val_preci

## Guardado y evaluación final

In [33]:
model_cargado.save('model_cargado_cbam_my_tf_fine_tuned_final_.keras')

In [34]:
evaluate = model_cargado.evaluate(test_generated)
print(evaluate)

146/146 ━━━━━━━━━━━━━━━━━━━━ 74s 507ms/step - accuracy: 0.8658 - loss: 0.6398 - precision: 0.8738 - recall: 0.8613
[0.6397889852523804, 0.8658124804496765, 0.8738467693328857, 0.8612660765647888]
